# 그램-슈미트와 QR 분해

> 선형대수 11강 · 직교성과 최소제곱법

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [그램-슈미트와 QR 분해](https://mioon1402.github.io/timeseriesdata/linalg/L11-qr.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 왜 직교 기저가 좋은가

## 1. 정규직교기저와 직교행렬 Q

## 2. 직교행렬의 성질

## 3. 그램-슈미트 과정

## 4. A = QR 분해

## 5. QR 로 최소제곱 풀기

## 6. numpy 로 확인하기

**11-1. 그램-슈미트 직접 구현**

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[1., 1., 2.],
              [1., 0., 1.],
              [1., 2., 0.]])

def 그램슈미트(A, tol=1e-12):
    Q = []
    for j in range(A.shape[1]):
        v = A[:, j].astype(float).copy()
        for q in Q:                        # 이미 만든 방향의 성분을 전부 뺀다
            v -= (q @ A[:, j]) * q
        n = np.linalg.norm(v)
        if n < tol:
            continue                       # 이미 있던 방향 = 종속 열, 건너뛴다
        Q.append(v / n)
    return np.column_stack(Q)

Q = 그램슈미트(A)
print("Q =")
print(Q)
print("\nQᵀQ =")
print(Q.T @ Q, "  ← 단위행렬이면 성공")

**11-2. R 은 그냥 QᵀA**

In [ ]:
R = Q.T @ A
print("R =")
print(R)
print("\n상삼각인가:", np.allclose(R, np.triu(R)))
print("Q @ R == A :", np.allclose(Q @ R, A))

**11-3. numpy 의 qr 과 비교**

In [ ]:
Q2, R2 = np.linalg.qr(A)
print("numpy Q =")
print(Q2)
print()
print("우리 Q 와 부호만 다른가:", np.allclose(np.abs(Q), np.abs(Q2)))
print()
print("둘 다 맞는 분해인가:")
print("  우리  Q@R   == A :", np.allclose(Q @ R, A))
print("  numpy Q2@R2 == A :", np.allclose(Q2 @ R2, A))

**11-4. 교과서 그램-슈미트가 무너지는 순간**

In [ ]:
# 거의 평행한 두 벡터
eps = 1e-8
나쁜A = np.array([[1., 1.],
                 [eps, 0.],
                 [0., eps]])

Q_ours = 그램슈미트(나쁜A)
Q_np, _ = np.linalg.qr(나쁜A)

print("우리 것  QᵀQ - I 의 최대 오차 :", np.abs(Q_ours.T @ Q_ours - np.eye(2)).max())
print("numpy 것 QᵀQ - I 의 최대 오차 :", np.abs(Q_np.T @ Q_np - np.eye(2)).max())
print()
print("→ 열이 거의 평행하면 '빼고 남은 것' 이 아주 작아져")
print("   반올림 오차가 상대적으로 커진다. 실무에서는 라이브러리를 쓰세요.")

**11-5. 직교행렬은 길이를 보존한다**

In [ ]:
rng = np.random.default_rng(0)
Qr, _ = np.linalg.qr(rng.normal(size=(4, 4)))       # 무작위 직교행렬

x = rng.normal(size=4)
y = rng.normal(size=4)

print("‖x‖   =", np.linalg.norm(x))
print("‖Qx‖  =", np.linalg.norm(Qr @ x), "  ← 같다")
print()
print("xᵀy    =", x @ y)
print("(Qx)ᵀ(Qy) =", (Qr @ x) @ (Qr @ y), "  ← 같다 (각도 보존)")
print()
print("det(Q) =", round(np.linalg.det(Qr), 6), " ← ±1 (회전이면 +1, 반사가 섞이면 −1)")

**11-6. QR 로 최소제곱 풀기**

In [ ]:
from scipy.linalg import solve_triangular

t = np.array([0., 1., 2., 3.])
y = np.array([1., 2., 2., 5.])
Am = np.column_stack([t, np.ones_like(t)])

Qm, Rm = np.linalg.qr(Am)
x_qr = solve_triangular(Rm, Qm.T @ y)      # R x = Qᵀb — 후진 대입

print("QR 로 푼 해     :", x_qr)
print("lstsq 의 해     :", np.linalg.lstsq(Am, y, rcond=None)[0])
print("정규방정식의 해 :", np.linalg.solve(Am.T @ Am, Am.T @ y))
print()
print("전부 같지만, 조건이 나쁜 문제에서는 차이가 납니다 ↓")

**11-7. 조건수가 제곱된다는 것**

In [ ]:
# 반데르몽드 행렬 — 다항식 회귀에서 나오며 조건이 아주 나쁘다
x_pts = np.linspace(0, 1, 8)
V_mat = np.vander(x_pts, 8)

c1 = np.linalg.cond(V_mat)
c2 = np.linalg.cond(V_mat.T @ V_mat)

print(f"cond(A)   = {c1:.3e}")
print(f"cond(AᵀA) = {c2:.3e}")
print(f"비율      = {c2/c1:.3e}   ≈ cond(A) = {c1:.3e}")
print()
print("→ AᵀA 를 만드는 순간 조건수가 제곱된다.")
print("   유효숫자가 절반으로 줄어드는 셈이라, 정규방정식을 직접 풀면 위험하다.")
print("   QR 이나 SVD(lstsq) 는 A 를 직접 다뤄 이 문제를 피한다.")

**11-8. 연습문제**

In [ ]:
# 문제 1. a1=[3,0], a2=[2,2] 를 그램-슈미트로 직교화해보세요.
#         손으로 예상한 뒤 코드로 확인하세요.

# 문제 2. 이미 직교인 두 벡터를 넣으면 그램-슈미트는 무엇을 하나요?

# 문제 3. a2 가 a1 의 상수배일 때(종속) 어떻게 되나요?

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1 — a1 은 x축, a2 에서 x성분을 빼면 y축만 남는다
A1 = np.column_stack([[3., 0.], [2., 2.]])
print("문제 1:\n", 그램슈미트(A1), "  ← (1,0), (0,1)")

# 문제 2 — 길이만 1로 바꿀 뿐 방향은 그대로
A2 = np.column_stack([[3., 0.], [0., 5.]])
print("\n문제 2:\n", 그램슈미트(A2), "  ← 방향 유지, 길이만 정규화")

# 문제 3 — 뺐더니 0 이 되어 열이 하나만 남는다
A3 = np.column_stack([[1., 2.], [3., 6.]])
Q3 = 그램슈미트(A3)
print("\n문제 3: Q 의 모양 =", Q3.shape, " ← 열이 하나뿐")
print("       원래 열이 2개인데 방향은 1개 = 랭크 1 (7강)")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)